In [59]:
from gurobipy import Model, GRB, quicksum
import pandas as pd
import numpy as np

# Problem 1: Cargo Plane Load Optimization

## 1. Indices and Sets
* $i = 1, \dots, 4$: Index for Cargo Types (C1, C2, C3, C4).
* $j = 1, 2, 3$: Index for Compartments (Front, Center, Rear).

## 2. Decision Variables
* $x_{i,j} \ge 0$: Tons of Cargo $i$ loaded into Compartment $j$.

## 3. Parameters
* $P_i$: Profit per ton of cargo $i$.
* $V_i$: Volume ($\text{m}^3$/ton) of cargo $i$.
* $A_i$: Max available weight (tons) of cargo $i$.
* $K_j^{wt}$: Weight capacity (tons) of compartment $j$.
* $K_j^{sp}$: Space capacity ($\text{m}^3$) of compartment $j$.

## 4. Objective Function

Maximize Total Profit:
$$\text{Maximize } Z = \sum_{i=1}^{4} \sum_{j=1}^{3} P_i \cdot x_{i,j}$$

## 5. Constraints

Let $W_j = \sum_{i=1}^{4} x_{i,j}$ (Total weight loaded in compartment $j$).

### Cargo Availability
$$\sum_{j=1}^{3} x_{i,j} \le A_i \quad \forall i = 1, \dots, 4$$
(Total amount of each cargo loaded cannot exceed its availability.)

### Compartment Capacity (Weight and Space)
$$W_j \le K_j^{wt} \quad \forall j = 1, 2, 3$$
$$\sum_{i=1}^{4} V_i \cdot x_{i,j} \le K_j^{sp} \quad \forall j = 1, 2, 3$$

### Weight Ratio Balance
Let $K_F^{wt}, K_C^{wt}, K_R^{wt}$ be the weight capacities for Front, Center, and Rear.

$$K_{\text{Center}}^{wt} W_{\text{Front}} - K_{\text{Front}}^{wt} W_{\text{Center}} = 0$$
$$K_{\text{Rear}}^{wt} W_{\text{Front}} - K_{\text{Front}}^{wt} W_{\text{Rear}} = 0$$

In [60]:
def setup_p1():
    variables = 12
    constraints_ineq = 10
    constraints_eq = 2

    obj = [310, 310, 310, 380, 380, 380, 350, 350, 350, 285, 285, 285]

    A_ineq = [[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
         [480, 0, 0, 650, 0, 0, 580, 0, 0, 390, 0, 0],
         [0, 480, 0, 0, 650, 0, 0, 580, 0, 0, 390, 0],
         [0, 0, 480, 0, 0, 650, 0, 0, 580, 0, 0, 390],
         [1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0],
         [0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
         [0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1]]
    
    A_eq = [[16, -10, 0, 16, -10, 0, 16, -10, 0, 16, -10, 0],
            [8, 0, -10, 8, 0, -10, 8, 0, -10, 8, 0, -10]]
    
    b_ineq = [18, 15, 23, 12, 6800, 8700, 5300, 10, 16, 8]

    b_eq = [0, 0]

    return variables, constraints_ineq, constraints_eq, A_ineq, A_eq, b_ineq, b_eq, obj

In [61]:
def solve_p1(variables, constraints_ineq, constraints_eq, A_ineq, A_eq, b_ineq, b_eq, obj):
    m = Model("A_1")
    m.setParam('OutputFlag', 0)

    x = [m.addVar(lb = 0.0, vtype = GRB.CONTINUOUS) for j in range(variables)]

    m.setObjective(quicksum(obj[j] * x[j] for j in range(variables)), sense = GRB.MAXIMIZE)

    for i in range(constraints_ineq):
        m.addConstr(quicksum(A_ineq[i][j] * x[j] for j in range(variables)) <= b_ineq[i])

    for i in range(constraints_eq):
        m.addConstr(quicksum(A_eq[i][j] * x[j] for j in range(variables)) == b_eq[i])

    m.optimize()

    return m, x

In [62]:
def report_p1(m, x):
    if m.Status == GRB.OPTIMAL:
        cargo_names = ["C1", "C2", "C3", "C4"]
        compartment_names = ["Front", "Center", "Rear"]
        
        solution_data = []
        for i in range(len(cargo_names)):
            row_data = []
            for j in range(len(compartment_names)):
                qty = x[3 * i + j].X 
                row_data.append(qty)
            solution_data.append(row_data)

        df = pd.DataFrame(solution_data, index=cargo_names, columns=compartment_names)

        print("\n" + "=" * 40)
        print("PROBLEM 1 REPORT")
        print("=" * 40)

        print("-" * 30)
        print(f"Maximum Profit: ${m.ObjVal:,.2f}")
        print("-" * 30)
        print("\n")

        print("-" * 30)
        print(f"Total Tons Accepted per Cargo")
        print("-" * 30)
        df['Total_Accepted'] = df.sum(axis=1)
        print(df[['Total_Accepted']])
        print("\n")

        print("-" * 30)
        print(f"Distribution Across Compartments")
        print("-" * 30)
        print(df[compartment_names])
        
    else:
        print("Model did not find an optimal solution.")

In [63]:
def run_p1():
    variables, constraints_ineq, constraints_eq, A_ineq, A_eq, b_ineq, b_eq, obj = setup_p1()
    model, values = solve_p1(variables, constraints_ineq, constraints_eq, A_ineq, A_eq, b_ineq, b_eq, obj)
    report_p1(model, values)

if __name__ == "__main__":
    run_p1()


PROBLEM 1 REPORT
------------------------------
Maximum Profit: $12,151.58
------------------------------


------------------------------
Total Tons Accepted per Cargo
------------------------------
    Total_Accepted
C1        0.000000
C2       15.000000
C3       15.947368
C4        3.052632


------------------------------
Distribution Across Compartments
------------------------------
    Front     Center  Rear
C1    0.0   0.000000   0.0
C2   10.0   0.000000   5.0
C3    0.0  12.947368   3.0
C4    0.0   3.052632   0.0


# Problem 2: Optimal Blending with Proportion Bounds

## 1. Indices and Sets
* $i = 1, 2, 3$: Index for Raw Materials (RM1, RM2, RM3).
* $j = 1, 2$: Index for Final Products (P1, P2).

## 2. Decision Variables
* $x_{i,j} \ge 0$: Tons of Raw Material $i$ used to produce Product $j$.

## 3. Parameters
* $C_i$: Unit Cost of Raw Material $i$.
* $A_i$: Tons of Raw Material $i$ available.
* $R_j$: Tons of Product $j$ required.
* $P_j$: Unit Selling Price of Product $j$.
* $R_{\text{Total}}$: Total Constant Revenue.
* $\text{LB}_{i,j}, \text{UB}_{i,j}$: Lower and Upper tonnage bounds for $x_{i,j}$, calculated from the required proportions multiplied by $R_j$.

## 4. Objective Function

Maximize Marginal Income (Revenue minus Total Raw Material Cost):

$$\text{Maximize } Z = R_{\text{Total}} + \sum_{i=1}^{3} \sum_{j=1}^{2} (-C_i) \cdot x_{i,j}$$

## 5. Constraints

### Raw Material Availability
$$\sum_{j=1}^{2} x_{i,j} \le A_i \quad \forall i = 1, 2, 3$$
(Total usage of each Raw Material cannot exceed its availability.)

### Production Requirement
$$\sum_{i=1}^{3} x_{i,j} = R_j \quad \forall j = 1, 2$$
(Total weight of ingredients for each product must equal the required quantity.)

### Proportion Bounds
$$\text{LB}_{i,j} \le x_{i,j} \le \text{UB}_{i,j} \quad \forall i = 1, 2, 3, \forall j = 1, 2$$

In [64]:
def setup_p2():
    variables = 6
    constraints_ineq = 3
    constraints_eq = 2

    lb = [240, 350, 60, 70, 120, 140]
    ub = [360, 420, 120, 280, 300, 210]

    A_ineq = [[1, 1, 0, 0, 0, 0],
              [0, 0, 1, 1, 0, 0],
              [0, 0, 0, 0, 1, 1]]
    
    A_eq = [[1, 0, 1, 0, 1, 0],
            [0, 1, 0, 1, 0, 1]]
    
    b_ineq = [2000, 1000, 500]

    b_eq = [600, 700]

    obj = [-1, -1, -1.5, -1.5, -3, -3]

    return variables, constraints_ineq, constraints_eq, lb, ub, A_ineq, A_eq, b_ineq, b_eq, obj

In [65]:
def solve_p2(variables, constraints_ineq, constraints_eq, lb, ub, A_ineq, A_eq, b_ineq, b_eq, obj):
    m = Model("A_2")
    m.setParam('OutputFlag', 0)

    x = []
    for j in range(variables):
        x.append(m.addVar(lb = lb[j], ub = ub[j], vtype = GRB.CONTINUOUS))

    m.setObjective(11600 + quicksum(x[j] * obj[j] for j in range(variables)), sense = GRB.MAXIMIZE)

    for i in range(constraints_ineq):
        m.addConstr(quicksum(A_ineq[i][j] * x[j] for j in range(variables)) <= b_ineq[i])

    for i in range(constraints_eq):
        m.addConstr(quicksum(A_eq[i][j] * x[j] for j in range(variables)) == b_eq[i])

    m.optimize()

    return m, x

In [66]:
def report_p2(m, x):
    if m.Status == GRB.OPTIMAL:
        raw_materials = ["RM1", "RM2", "RM3"]
        products = ["P1", "P2"]
        
        solution_data = []
        for i in range(len(raw_materials)):
            qty_p1 = x[2 * i + 0].X 
            qty_p2 = x[2 * i + 1].X 
            row_data = [qty_p1, qty_p2]
            solution_data.append(row_data)

        df = pd.DataFrame(solution_data, index=raw_materials, columns=products)
        
        print("\n" + "=" * 40)
        print("PROBLEM 2 REPORT")
        print("=" * 40)
        
        print("-" * 30)
        print(f"Maximum Profit: €{m.ObjVal:,.2f}")
        print("-" * 30)

        print("-" * 30)
        print("Optimal Blend Plan:")
        print("-" * 30)
        
        df['Total Used'] = df.sum(axis=1)
        
        df.loc['Total Product Weight'] = df.sum(axis=0)
        
        print(df.round(2))
        
    else:
        print("Model did not find an optimal solution.")

In [67]:
def run_p2():
    variables, constraints_ineq, constraints_eq, lb, ub, A_ineq, A_eq, b_ineq, b_eq, obj = setup_p2()
    model, values = solve_p2(variables, constraints_ineq, constraints_eq, lb, ub, A_ineq, A_eq, b_ineq, b_eq, obj)
    report_p2(model, values)

if __name__ == "__main__":
    run_p2()


PROBLEM 2 REPORT
------------------------------
Maximum Profit: €9,650.00
------------------------------
------------------------------
Optimal Blend Plan:
------------------------------
                         P1     P2  Total Used
RM1                   360.0  420.0       780.0
RM2                   120.0  140.0       260.0
RM3                   120.0  140.0       260.0
Total Product Weight  600.0  700.0      1300.0


# Problem 3: The Knapsack Problem

## 1. Indices and Sets
* $i = 1, \dots, 12$: Index for the valuable items.

## 2. Decision Variables
* $\mathbf{x_{i}} \in \{0, 1\}$: The **binary** decision variable for item $i$.
    * $x_{i} = 1$ if item $i$ is selected.
    * $x_{i} = 0$ if item $i$ is not selected.

## 3. Parameters
* $P_i$: Price/Value (in euros) of item $i$.
* $W_i$: Weight (in kg) of item $i$.
* $K$: Maximum knapsack capacity.

## 4. Objective Function

Maximize the Total Value of selected items:
$$\text{Maximize } Z = \sum_{i=1}^{12} P_i \cdot x_{i}$$

## 5. Constraints

### Weight Capacity
$$\sum_{i=1}^{12} W_i \cdot x_{i} \le K$$

### Binary Restriction
$$x_{i} \in \{0, 1\} \quad \forall i = 1, \dots, 12$$

In [68]:
def setup_p3():
    variables = 12
    constraints = 1

    obj = [95, 58, 52, 96, 48, 84, 94, 56, 91, 70, 54, 61]

    A_ineq = [[180, 91, 131, 211, 142, 137, 204, 176, 98, 183, 157, 108]]

    b_ineq = [900]

    return variables, constraints, A_ineq, b_ineq, obj

In [69]:
def solve_p3(variables, constraints, A_ineq, b_ineq, obj):
    m = Model("A_3")
    m.setParam('OutputFlag', 0)

    x = [m.addVar(vtype = GRB.BINARY) for j in range(variables)]

    m.setObjective(quicksum(obj[j] * x[j] for j in range(variables)), sense = GRB.MAXIMIZE)

    for i in range(constraints):
        m.addConstr(quicksum(A_ineq[i][j] * x[j] for j in range(variables)) <= b_ineq[i])

    m.optimize()

    return m, x

In [70]:
def report_p3(m, x):
    if m.Status == GRB.OPTIMAL:
        item_indices = list(range(1, 13))
        
        selected_items = []
        
        W_i = [180, 91, 131, 211, 142, 137, 204, 176, 98, 183, 157, 108]
        P_i = [95, 58, 52, 96, 48, 84, 94, 56, 91, 70, 54, 61]
        
        total_weight_used = 0
        
        for i in range(len(item_indices)):
            if x[i].X > 0.99:
                selected_items.append(item_indices[i])
                total_weight_used += W_i[i]
        
        print("\n" + "=" * 40)
        print("PROBLEM 3 REPORT")
        print("=" * 40)

        print("-" * 30)
        print(f"Maximum Profit: €{m.ObjVal:,.2f}")
        print("-" * 30)
        print("\n")

        print("-" * 30)
        print(f"Optimal Item Selection")
        print("-" * 30)
        print(f"Items Selected: {selected_items}")
        print(f"Total Weight Used: {total_weight_used:.2f} kg out of 900 kg")
        print("-" * 30)
        
    else:
        print("Model did not find an optimal solution.")

In [71]:
def run_p3():
    variables, constraints, A_ineq, b_ineq, obj = setup_p3()
    model, values = solve_p3(variables, constraints, A_ineq, b_ineq, obj)
    report_p3(model, values)

if __name__ == "__main__":
    run_p3()


PROBLEM 3 REPORT
------------------------------
Maximum Profit: €495.00
------------------------------


------------------------------
Optimal Item Selection
------------------------------
Items Selected: [1, 2, 4, 7, 9, 12]
Total Weight Used: 892.00 kg out of 900 kg
------------------------------


# Problem 4: Multi-Product Production Planning with Data Handling

## 1. Indices and Sets
* $i = 1, \dots, n$: Index for the different products to be manufactured.

## 2. Decision Variables
* $\mathbf{x_{i}} \ge 0$: Weekly production quantity (units) of product $i$.

## 3. Parameters
* $P_i$: Profit per unit produced (€/unit) of product $i$.
* $T_i$: Production time required per unit (hours/unit) of product $i$.
* $R_i$: Raw material required per unit (kg/unit) of product $i$.
* $D_i$: Maximum weekly demand (units) for product $i$.
* $T_{\text{max}}$: Total available production time.
* $R_{\text{max}}$: Total available raw material.

## 4. Objective Function

Maximize the total weekly profit:
$$\text{Maximize } Z = \sum_{i=1}^{n} P_i \cdot x_{i}$$

## 5. Constraints

### Production Time Constraint
$$\sum_{i=1}^{n} T_i \cdot x_{i} \le T_{\text{max}}$$

### Raw Material Constraint
$$\sum_{i=1}^{n} R_i \cdot x_{i} \le R_{\text{max}}$$

### Market Demand Constraint
$$x_{i} \le D_i \quad \forall i = 1, \dots, n$$

### Non-negativity
$$x_{i} \ge 0 \quad \forall i = 1, \dots, n$$

In [72]:
def setup_p4():
    global_df = pd.read_csv('Data_Exercise_4.csv', header=None, nrows=3, index_col=0)
    
    T_max = global_df.loc['TimeCap', 1]
    R_max = global_df.loc['MaterialCap', 1]
    num_products = int(global_df.loc['Nproducts', 1])

    df = pd.read_csv('Data_Exercise_4.csv', header=4).set_index('Products').T
    
    variables = num_products
    obj = df['Profit'].tolist() 
    
    time_coeffs = df['Time'].tolist()
    material_coeffs = df['Material'].tolist()
    demand_limits = df['Demand'].tolist()
    
    A_ineq = [time_coeffs, material_coeffs] 
    
    A_ineq.extend(np.identity(num_products).tolist()) 

    b_ineq = [T_max, R_max] + demand_limits
    
    constraints_ineq = len(A_ineq)
    
    return variables, constraints_ineq, A_ineq, b_ineq, obj, df

In [73]:
def solve_p4(variables, constraints, A_ineq, b_ineq, obj, df):
    m = Model("A_4")
    m.setParam('OutputFlag', 0)

    x = [m.addVar(lb = 0.0, vtype = GRB.CONTINUOUS) for j in range(variables)]

    m.setObjective(quicksum(obj[j] * x[j] for j in range(variables)), sense = GRB.MAXIMIZE)

    for i in range(constraints):
        m.addConstr(quicksum(A_ineq[i][j] * x[j] for j in range(variables)) <= b_ineq[i])

    m.optimize()

    return m, x

In [74]:
def report_p4(m, x, df):
    if m.Status == GRB.OPTIMAL:
        
        optimal_production = [x[j].X for j in range(len(x))]
        
        df['Optimal Production Units'] = optimal_production
        df['Optimal Profit'] = df['Optimal Production Units'] * df['Profit']
        
        print("\n" + "=" * 40)
        print("PROBLEM 4 REPORT")
        print("=" * 40)

        print("-" * 30)
        print(f"Maximum Weekly Profit: €{m.ObjVal:,.2f}")
        print("-" * 30)
        
        df.to_csv('Data_Exercise_4.csv', index=False)
        print("Results successfully written back to Data_Exercise_4.csv!")
        
    else:
        print("Model did not find an optimal solution.")

In [75]:
def run_p4():
    variables, constraints, A_ineq, b_ineq, obj, df = setup_p4()
    model, values = solve_p4(variables, constraints, A_ineq, b_ineq, obj, df)
    report_p4(model, values, df)

if __name__ == "__main__":
    run_p4()


PROBLEM 4 REPORT
------------------------------
Maximum Weekly Profit: €700.00
------------------------------
Results successfully written back to Data_Exercise_4.csv!
